In [ ]:
import json
import pandas as pd


import cv2 
from typing import TypedDict

import matplotlib.pyplot as plt

import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.insert(0, str(ROOT))

from src.config.paths import *
from src.data.plots import image_category

ANALYSIS_DIR = CSV_DIR / "data_analysis"
IMAGE_DIR = IMAGE_DIR / "data_analysis"
PLOT_DIR = PLOT_DIR / "data_analysis"

In [ ]:
df = pd.read_csv(CSV_DIR / "data/metadata.csv")
df["boxes"] = df["boxes"].apply(json.loads)

In [ ]:
class ImageProperties(TypedDict):
    width: int
    height: int

    brightness: float
    sharpness: float
    contrast: float

    split: str
    category: str
    path: str
    

In [ ]:
stats = []
df["category"] = df["boxes"].apply(image_category)

for row in df.itertuples():
    img = cv2.imread(str(DATASET_DIR / row.path), cv2.IMREAD_GRAYSCALE)

    h, w = img.shape

    stats.append(ImageProperties(
        width=w,
        height=h,
        brightness=img.mean(),
        sharpness=cv2.Laplacian(img, cv2.CV_64F).var(),
        contrast=img.std(),
        split=row.split,
        category=row.category,
        path=row.path
    ))

stats = pd.DataFrame(stats)
stats.to_csv(ANALYSIS_DIR/"image_properties.csv", index=False)

stats

In [ ]:
plt.figure(figsize=(10, 5))

plt.scatter(
    stats["width"],
    stats["height"],
)

plt.xlabel("Width")
plt.ylabel("Height")

plt.show()

#TODO find a way to plot the 2 trendlines i can see in this data of the aspect ratios

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 12))

for ax, prop in zip(axes, ["brightness", "contrast", "sharpness"]):
    ax.boxplot(
        [
            stats[prop][stats["category"] == "neither"],
            stats[prop][stats["category"] == "both"],
            stats[prop][stats["category"] == "smoke_only"],
            stats[prop][stats["category"] == "fire_only"],
        ],
        tick_labels=["Neither", "Both", "Smoke", "Fire"]
    )
    ax.set_title(prop)

plt.savefig(PLOT_DIR / "image_prop_box_plots.svg")
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(18, 12))

for ax, prop in zip(axes, ["brightness", "contrast", "sharpness"]):
    ax.boxplot(
        [
            stats[prop][stats["split"] == split]          
            for split in ["train", "val", "test"]
        ],
        tick_labels=["Train", "Val", "Test"],
        showfliers=False
    )
    ax.set_title(prop)
    ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.savefig(PLOT_DIR / "image_props_by_split.svg")
plt.show()